In [1]:
# load FAQ data only for llm course
from ingest import load_faq_data
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

documents = documents_llm

In [2]:
# define the required structure for the LLM response
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [3]:


import json
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)


data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions on the internet. Not too formal, not too short, not too long.
""".strip()


In [4]:
# TEST - Generate 5 questions for the first FAQ record
from evaluation_utils import llm_structured

doc = documents[0]
print(json.dumps(doc))

parsed_response, usage = llm_structured(openai_client, data_gen_instructions, json.dumps(doc), Questions, os.getenv("OPENAI_MODEL"))

print(parsed_response)

{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."}
questions=["I'm a bit late to the party, is it still possible to sign up?", 'Can I enroll in the class even though it already started?', 'Is there still a chance for me to get a certificate if I join now?', 'I just found out about this zoomcamp, am I too late to participate?', 'If I start the course today, what do I need to do to get certified?']


In [5]:
from evaluation_utils import calculate_token_costs
print(calculate_token_costs(usage, os.getenv("OPENAI_MODEL")))


{'input_cost': 3.7800000000000004e-05, 'output_cost': 4.2800000000000004e-05, 'total_cost': 8.060000000000001e-05}


In [6]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        os.getenv("OPENAI_MODEL"),
        max_retries=3
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [7]:
# from tqdm.auto import tqdm

# ground_truth = []
# usages = []

# for doc in tqdm(documents[:5]):
#     records, usage = generate_ground_truth(doc)
#     ground_truth.extend(records)
#     usages.append(usage)

In [8]:
# Generate ground truth as parallel processing
# Only doing first 50 because of IK rate limits
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

  0%|          | 0/153 [00:00<?, ?it/s]

765

In [10]:
total_cost = 0.0

for usage in usages:
    cost = calculate_token_costs(usage, os.getenv("OPENAI_MODEL"))
    total_cost = total_cost + cost["total_cost"]

print(total_cost)

0.017848399999999997


In [12]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)